NA CAMADA SILVER SÃO FEITOS BOA PARTE DOS TRATAMENTOS, ALTERAÇÕES DE NOMES DE COLUNAS E FILTROS


PRIMEIRAMENTE VAMOS IMPORTAR OS NOTEBOOKS QUE ESTÃO PRESENTES NA CAMADA BRONZE


In [0]:
import pyspark.sql.functions as F

In [0]:
df_bronze_dm_produtos = spark.table("ecommerce.bronze.dm_categoria_produtos_traducao")
df_bronze_ft_avaliacoes = spark.table("ecommerce.bronze.ft_avaliacoes_pedidos")
df_bronze_ft_consumidores = spark.table("ecommerce.bronze.ft_consumidores")
df_bronze_ft_geolocalizacao = spark.table("ecommerce.bronze.ft_geolocalizacao")
df_bronze_ft_itens_pedidos = spark.table("ecommerce.bronze.ft_itens_pedidos")
df_bronze_ft_pagamentos_pedidos = spark.table("ecommerce.bronze.ft_pagamentos_pedidos")
df_bronze_ft_pedidos = spark.table("ecommerce.bronze.ft_pedidos")
df_bronze_ft_produtos = spark.table("ecommerce.bronze.ft_produtos")
df_bronze_ft_vendedoras = spark.table("ecommerce.bronze.ft_vendedores")
df_bronze_dm_categoria_produtos_traducao = spark.table("ecommerce.bronze.dm_categoria_produtos_traducao")

In [0]:
display(df_bronze_ft_consumidores.limit(5))

AGORA IREI ALTERAR OS NOMES DAS MINHAS COLUNAS CONFORME O SOLICITADO

In [0]:
df_silver_ft_consumidores = df_bronze_ft_consumidores.select(
    F.col("customer_id").alias("id_consumidor"),
    F.col("customer_zip_code_prefix").alias("prefixo_cep"),
    F.col('customer_city').alias("cidade"),
    F.col('customer_state').alias("estado")
)

In [0]:
display(
    df_silver_ft_consumidores.select(
        F.count(
            F.when(F.col("id_consumidor").isNull(), 1)
        ).alias("qtd_nulos_id_consumidor")
    )
)

VAMOS CONTAR A QUANTIDADE DE NULOS EXISTENTES E DUPLICADOS NA COLUNA DE ID_CONSUMIDOR

In [0]:


qtd_nulos = df_silver_ft_consumidores.filter(F.col("id_consumidor").isNull()).count()

qtd_duplicados = (
    df_silver_ft_consumidores
    .groupBy("id_consumidor")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Nulos: {qtd_nulos}")
print(f"Duplicados: {qtd_duplicados}")


In [0]:
df_silver_ft_consumidores = (
    df_silver_ft_consumidores
    .withColumn("cidade", F.upper(F.col("cidade")))
    .withColumn("estado", F.upper(F.col("estado")))
)

In [0]:
display(df_silver_ft_consumidores.limit(3))

AGORA FAREMOS AS VERIFICAÇÕES PARA O ft_pedidos

In [0]:
display(df_bronze_ft_pedidos.limit(3))

In [0]:
df_silver_ft_pedidos = df_bronze_ft_pedidos.select(
    F.col("order_id").alias("id_pedido"),
    F.col("customer_id").alias("id_consumidor"),
    F.col("order_status").alias("status"),
    F.col("order_purchase_timestamp").alias("pedido_compra_timestamp"),
    F.col("order_approved_at").alias("pedido_aprovado_timestamp"),
    F.col("order_delivered_carrier_date").alias("pedido_carregado_timestamp"),
    F.col("order_delivered_customer_date").alias("pedido_entregue_timestamp"),
    F.col("order_estimated_delivery_date").alias("pedido_estimativa_entrega_timestamp")
)

In [0]:
qtd_nulos = df_silver_ft_pedidos.filter(F.col("id_consumidor").isNull()).count()

qtd_duplicados = (
    df_silver_ft_pedidos
    .groupBy("id_consumidor")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Nulos: {qtd_nulos}")
print(f"Duplicados: {qtd_duplicados}")



In [0]:
df_silver_ft_pedidos = df_silver_ft_pedidos.withColumn(
    "status",
    F.when(F.col("status") == "canceled", "cancelado")
    .when(F.col("status") == "shipped", "enviado")
    .when(F.col("status") == "processing", "em processamento")
    .when(F.col("status") == "unavailable", "indisponivel")
    .when(F.col("status") == "invoiced", "faturado")
    .when(F.col("status") == "approved", "aprovado")
    .when(F.col("status") == "delivered", "entregue")
    .when(F.col("status") == "created", "criado")
    .otherwise("nao_classificado")
)

In [0]:
display(df_silver_ft_pedidos.limit(5))


VERIFICANDO SE EXISATEM CASOS EM QUE NÃO TENHO CLASSIFICAÇÃO

In [0]:
df_silver_ft_pedidos.filter(F.col("status") == "nao_classificado").display()

In [0]:
df_silver_ft_pedidos.display()

AGORA IREI CRIAR AS COLUNAS NECESSÁRIAS

In [0]:
df_silver_ft_pedidos = (
    df_silver_ft_pedidos
    .withColumn(
        "tempo_entrega_dias",
        F.datediff(F.col("pedido_entregue_timestamp"), F.col("pedido_compra_timestamp"))
    )
    .withColumn(
        "tempo_entrega_estimado_dias",
        F.datediff(F.col("pedido_estimativa_entrega_timestamp"), F.col("pedido_compra_timestamp"))
    )
    .withColumn(
        "diferenca_entrega_dias",
        F.col("tempo_entrega_estimado_dias") - F.col("tempo_entrega_dias")
    )
    .withColumn(
        "entregue_no_prazo",
        F.when(F.col("diferenca_entrega_dias") <= 0, "Sim")
         .when(F.col("diferenca_entrega_dias") > 0, "Não")
         .otherwise("Não Entregue")
    )
)

display(df_silver_ft_pedidos)
